In [ ]:
from moseplib.data import pointcloud_processing, timeseries_processing

from pathlib import Path

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
DATA_DIR: Path = Path("/datalocal/chg/MOLISENSext") / "ViF_Roof" / "data"

BAG_NAME_REFERENCE: str = "molisens_met_2023_08_07-15_36_45_converted"
BAG_NAME_RAIN: str = "molisens_met_2023_08_29-06_04_46_converted"


## Load data weather stations (WS)

In [3]:
df_reference = timeseries_processing.load_timeseries(
    DATA_DIR,
    BAG_NAME_REFERENCE,
    topics="/sensing/aws/ws100_measurements",
    safe_parquet=True,
    verbose=True,
)

df_rain = timeseries_processing.load_timeseries(
    DATA_DIR,
    BAG_NAME_RAIN,
    topics="/sensing/aws/ws100_measurements",
    safe_parquet=True,
    verbose=True,
)

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/processed/weather_df_molisens_met_2023_08_07-15_36_45_converted.parquet

Found parquet files, loading timeseries data...

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/processed/weather_df_molisens_met_2023_08_29-06_04_46_converted.parquet

Found parquet files, loading timeseries data...

In [4]:
df_rain

Catgegory           temperature precipitation                           \
Parameter            r2s_heater      absolute differential  type  code   
Timestamp                                                                
2023-08-29 04:05:00   15.416394     24.869999          0.0  60.0  61.0   
2023-08-29 04:05:01   15.416394     24.869999          0.0  60.0  61.0   
2023-08-29 04:05:02   15.322683     24.869999          0.0  60.0  61.0   
2023-08-29 04:05:03   15.322683     24.869999          0.0  60.0  61.0   
2023-08-29 04:05:04   15.275838     24.869999          0.0  60.0  61.0   
...                         ...           ...          ...   ...   ...   
2023-08-29 05:04:41   15.510098     32.220001          0.0   0.0  61.0   
2023-08-29 05:04:42   15.275838     32.220001          0.0   0.0  61.0   
2023-08-29 05:04:43   15.135283     32.220001          0.0   0.0  61.0   
2023-08-29 05:04:44   15.041578     32.220001          0.0   0.0  61.0   
2023-08-29 05:04:45   14.947867     32.220001          0.0   0.0  61.0   

Catgegory                                            \
Parameter           intensity_hour intensity_minute   
Timestamp                                             
2023-08-29 04:05:00           0.12            0.002   
2023-08-29 04:05:01           0.12            0.002   
2023-08-29 04:05:02           0.12            0.002   
2023-08-29 04:05:03           0.12            0.002   
2023-08-29 04:05:04           0.12            0.002   
...                            ...              ...   
2023-08-29 05:04:41           0.00            0.000   
2023-08-29 05:04:42           0.00            0.000   
2023-08-29 05:04:43           0.00            0.000   
2023-08-29 05:04:44           0.00            0.000   
2023-08-29 05:04:45           0.00            0.000   

Catgegory                                                      \
Parameter           total_precipitation_particles total_drops   
Timestamp                                                       
2023-08-29 04:05:00                           0.0         0.0   
2023-08-29 04:05:01                           0.0         0.0   
2023-08-29 04:05:02                           0.0         0.0   
2023-08-29 04:05:03                           0.0         0.0   
2023-08-29 04:05:04                           0.0         0.0   
...                                           ...         ...   
2023-08-29 05:04:41                           0.0         0.0   
2023-08-29 05:04:42                           0.0         0.0   
2023-08-29 05:04:43                           0.0         0.0   
2023-08-29 05:04:44                           0.0         0.0   
2023-08-29 05:04:45                           0.0         0.0   

Catgegory                              ...          wind                \
Parameter           drizzle_particles  ... direction_min direction_max   
Timestamp                              ...                               
2023-08-29 04:05:00               0.0  ...           0.0    340.285400   
2023-08-29 04:05:01               0.0  ...           0.0    340.285400   
2023-08-29 04:05:02               0.0  ...           0.0    340.285400   
2023-08-29 04:05:03               0.0  ...           0.0    340.285400   
2023-08-29 04:05:04               0.0  ...           0.0    340.285400   
...                               ...  ...           ...           ...   
2023-08-29 05:04:41               0.0  ...           0.0      4.087067   
2023-08-29 05:04:42               0.0  ...           0.0      4.087067   
2023-08-29 05:04:43               0.0  ...           0.0      4.087067   
2023-08-29 05:04:44               0.0  ...           0.0      4.087067   
2023-08-29 05:04:45               0.0  ...           0.0      4.087067   

Catgegory                                                              \
Parameter           direction_vector_avg direction_fast value_quality   
Timestamp                                                               
2023-08-29 04:05:00      

## Load point cloud data (resampled)

### Temporal Resampling Routine Documentation
High-temporal-resolution pointcloud datasets (10 Hz) are aggregated by a resampling routine into 1-minute intervals to reduce computational complexity, using resample_dataset to group pointclouds by time and compute statistics like mean (default), standard deviation ("std"), and sum ("sum") for each bin. During aggregation, if multiple pointclouds contain points with identical coordinates, statistics (e.g., averaging intensities for mean or summing values for sum) are applied across them; if coordinates are unique, the point is simply added unchanged, as statistics on a single point have no effect. This process condenses high-frequency data into coarser temporal representations while preserving spatial and attribute information, resulting in a dictionary of datasets ("mean", "std", "sum") that are saved to Parquet files for persistence and reloaded for analysis.

In [5]:
TOPICS = {
    "lid_pts": "/sensing/lidar/points",
    "lid_pts2": "/sensing/lidar/points2",
    "rad_pts": "/sensing/radar/points",
}

In [6]:
dataset_reference = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME_REFERENCE, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"]
)
dataset_reference_2 = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME_REFERENCE, topic=TOPICS["lid_pts2"], verbose=True, invert_axes=["x", "y"]
)
dataset_rain = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME_RAIN, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"]
)
dataset_rain_2 = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME_RAIN, topic=TOPICS["lid_pts2"], verbose=True, invert_axes=["x", "y"]
)

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_07-15_36_45_conver
ted

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_07-15_36_45_conver
ted

 start =    2023-08-07 13:36:48.072408
 end =      2023-08-07 13:39:47.651742
 duration = 0:02:59.579334
 length =   1795
 avg frequency =  10.00 Hz

Inverting axes:
['x', 'y']

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points2/molisens_met_2023_08_07-15_36_45_conve
rted

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points2/molisens_met_2023_08_07-15_36_45_conve
rted

 start =    2023-08-07 13:36:47.984902
 end =      2023-08-07 13:39:47.651742
 duration = 0:02:59.666840
 length =   1798
 avg frequency =  10.01 Hz

...

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_29-06_04_46_conver
ted

 start =    2023-08-29 04:05:19.853657
 end =      2023-08-29 05:04:46.069975
 duration = 0:59:26.216318
 length =   35659
 avg frequency =  10.00 Hz

Inverting axes:
['x', 'y']

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points2/molisens_met_2023_08_29-06_04_46_conve
rted

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points2/molisens_met_2023_08_29-06_04_46_conve
rted

 start =    2023-08-29 04:05:19.853657
 end =      2023-08-29 05:04:46.069975
 duration = 0:59:26.216318
 length =   35664
 avg frequency =  10.00 Hz

Inverting axes:
['x', 'y']